# Oracle projection-safety test for direct SG guidance

This notebook tests whether direct space-group projection in DiffCSP++ `k` space is safe for the actual KLDMPlus data chart.

The diagnostic is deliberately strict:

- Start from a ground-truth training sample `(F0, k0, A, G)`.
- Compute the SG-projected lattice `k_proj = Pi_G(k0)`.
- Keep fractional coordinates and atom types fixed.
- Evaluate `(F0, k_rho, A)` against the original `(F0, k0, A)` using the normal CSP evaluator.

If hard projection breaks StructureMatcher or causes large length, angle, or volume drift, then a hard direct SG projection is unsafe. Soft guidance may still be safe if small `rho` values preserve match rate and CSP metrics.

## What this answers

For each interpolation strength

```text
k_rho = (1 - rho) * k0 + rho * Pi_G(k0)
```

we measure:

- CSP validity, match rate, RMSE, fractional RMSE.
- Lattice length RMSE, angle RMSE, and volume relative error.
- Detected family/space-group agreement.
- Wyckoff letter agreement where pymatgen can infer it.
- Projection benefit: how much `|k - Pi_G(k)|` shrinks.

This is the direct answer to: "If my SG loss pulls the predicted lattice toward the projector while coordinates stay in the old fractional chart, how much damage does that cause?"

In [1]:
from __future__ import annotations

import csv
import json
import sys
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
import yaml
from IPython.display import display

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "src" / "kldmPlus").exists():
    raise RuntimeError(f"Could not locate repo root from cwd={Path.cwd()}")

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from kldmPlus.data import CSPTask, resolve_data_root
from kldmPlus.diffusionModels.continuous import ContinuousVPDiffusion
from kldmPlus.sample_evaluation.sample_evaluation import (
    aggregate_csp_reconstruction_metrics,
    evaluate_csp_reconstruction,
)
from kldmPlus.symmetry.latticeSymmetry import LatticeSymmetry

CONFIG_PATH = ROOT / "configs" / "kldm_plus" / "mp_20" / "mp20_diffcsp_k_x0_soft_lattice_laptop.yaml"
cfg = yaml.safe_load(CONFIG_PATH.read_text())
cfg

MODELS_PROJECT_ROOT: /workspace/.venv/lib/python3.11/site-packages/mattergen


{'experiment_name': 'plus_mp20_diffcsp_k_x0_soft_lattice_laptop',
 'sampler_config': '../../samplers/predictor_corrector.yaml',
 'dataset': {'task': 'csp',
  'name': 'mp20',
  'lattice_representation': 'diffcsp_k',
  'root': None,
  'batch_size': 32,
  'num_workers': 0,
  'pin_memory': False,
  'train_subset_fraction': 0.1,
  'train_subset_seed': 2002,
  'train_subset_strategy': 'balanced_space_group'},
 'time_sampler': {'type': 'uniform'},
 'model': {'eps': 1e-06,
  'lambda_l': 1.0,
  'lattice_parameterization': 'x0',
  'lattice_diffusion_type': 'VP',
  'lattice_representation': 'diffcsp_k',
  'lattice_sg_lambda': 1,
  'lattice_sg_normalize': True,
  'lattice_sg_time_weight': 'alpha_squared',
  'lattice_orbit_metric_max_candidates': 512,
  'wrapped_normal_K': 3,
  'tdm_n_sigmas': 512,
  'tdm_compute_sigma_norm': True,
  'tdm_velocity_scale': 0.15915494309189535,
  'tdm_sigma_norm_estimator': 'quadrature',
  'tdm_sigma_norm_density_K': 20,
  'tdm_sigma_norm_grid_points': 1025,
  'tdm_s

In [2]:
DEVICE = torch.device("cpu")
torch.set_grad_enabled(False)

dataset_cfg = cfg["dataset"]
model_cfg = cfg["model"]

task = CSPTask(
    dataset_name=dataset_cfg["name"],
    lattice_parameterization=model_cfg["lattice_parameterization"],
    lattice_representation=dataset_cfg["lattice_representation"],
)
root = resolve_data_root(dataset_cfg.get("root"))
lattice_transform = task.make_lattice_transform(root=root, download=False)
sym = LatticeSymmetry(eps=float(model_cfg.get("eps", 1e-8))).to(DEVICE)

print("repo", ROOT)
print("data root", root)
print("config", CONFIG_PATH)
print("dataset", dataset_cfg["name"])
print("lattice_representation", dataset_cfg["lattice_representation"])
print("lattice_parameterization", model_cfg["lattice_parameterization"])
print("lattice_sg_lambda", model_cfg.get("lattice_sg_lambda"))
print("lattice_sg_time_weight", model_cfg.get("lattice_sg_time_weight"))
print("transform", type(lattice_transform).__name__)

repo /workspace
data root /workspace/data
config /workspace/configs/kldm_plus/mp_20/mp20_diffcsp_k_x0_soft_lattice_laptop.yaml
dataset mp20
lattice_representation diffcsp_k
lattice_parameterization x0
lattice_sg_lambda 1
lattice_sg_time_weight alpha_squared
transform DiffCSPKContinuousIntervalLattice


## Configuration

`MAX_SAMPLES` defaults to a bounded smoke-test size so the notebook is usable on CPU. Set it to `None` when you want the full train-set answer.

The `rho` grid includes the rough soft-guidance pulls discussed earlier plus hard projection at `rho = 1.0`.

In [3]:
SPLIT = "train"
MAX_SAMPLES: int | None = 512  # Set to None for the full split.
DOWNLOAD_DATA = False

RHOS = [0.0, 0.03, 0.05, 0.09, 0.15, 0.25, 0.40, 1.00]

# Normal CSP evaluator tolerances used by sample_evaluation.py.
STOL = 0.5
ANGLE_TOL = 10.0
LTOL = 0.3
VALIDITY_CUTOFF = 0.5
SG_SYMPREC = 1e-2
SG_ANGLE_TOLERANCE = 5.0

OUTPUT_DIR = ROOT / "notebooks" / "outputs" / "oracle_projection_safety"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("split", SPLIT)
print("max_samples", MAX_SAMPLES)
print("rhos", RHOS)
print("output_dir", OUTPUT_DIR)

split train
max_samples 512
rhos [0.0, 0.03, 0.05, 0.09, 0.15, 0.25, 0.4, 1.0]
output_dir /workspace/notebooks/outputs/oracle_projection_safety


## Lambda-to-rho intuition

For a single constrained component, a simplified quadratic objective

```text
(k_hat - k0)^2 + lambda_sg * w(t) * (k_hat - Pi_G(k_hat))^2
```

behaves approximately like a pull toward the projected value:

```text
rho(t) = lambda_sg * w(t) / (1 + lambda_sg * w(t))
```

This cell estimates the mean time weight for the currently configured SG weighting. Since your configs now use `alpha_squared`, the notebook estimates `E[alpha(t)^2]` from the VP diffusion instead of assuming only `E[(1-t)^2] = 1/3`.

In [4]:
def estimate_mean_sg_time_weight(kind: str, *, n: int = 20000) -> float:
    kind = str(kind)
    if kind == "none":
        return 1.0
    if kind == "quadratic_late":
        return 1.0 / 3.0
    if kind == "alpha_squared":
        vp = ContinuousVPDiffusion(
            eps=float(model_cfg.get("eps", 1e-6)),
            parameterization=str(model_cfg.get("lattice_parameterization", "x0")),
        )
        t = torch.linspace(0.0, 1.0, int(n))
        return float(vp.alpha(t).square().mean().item())
    raise ValueError(f"Unknown SG time weight: {kind}")


def lambda_to_rho(lambda_sg: float, mean_time_weight: float) -> float:
    pull = float(lambda_sg) * float(mean_time_weight)
    return pull / (1.0 + pull)


def rho_to_lambda(rho: float, mean_time_weight: float) -> float:
    rho = float(rho)
    if rho >= 1.0:
        return float("inf")
    return rho / (float(mean_time_weight) * (1.0 - rho))

mean_time_weight = estimate_mean_sg_time_weight(model_cfg.get("lattice_sg_time_weight", "none"))
current_lambda = float(model_cfg.get("lattice_sg_lambda", 0.0))

lambda_table = pd.DataFrame(
    [
        {
            "lambda_sg": lam,
            "mean_time_weight": mean_time_weight,
            "approx_mean_rho": lambda_to_rho(lam, mean_time_weight),
        }
        for lam in sorted(set([0.05, 0.1, 0.3, 1.0, 2.0, current_lambda]))
    ]
)

rho_table = pd.DataFrame(
    [
        {
            "rho": rho,
            "lambda_needed_at_mean_weight": rho_to_lambda(rho, mean_time_weight),
        }
        for rho in RHOS
    ]
)

print("mean_time_weight", mean_time_weight)
display(lambda_table)
display(rho_table)

mean_time_weight 0.2760066092014313


,lambda_sg,mean_time_weight,approx_mean_rho
0,0.05,0.276007,0.013612
1,0.10,0.276007,0.026859
2,0.30,0.276007,0.076470
3,1.00,0.276007,0.216305
4,2.00,0.276007,0.355676


,rho,lambda_needed_at_mean_weight
0,0.00,0.000000
1,0.03,0.112055
2,0.05,0.190690
3,0.09,0.358329
4,0.15,0.639371
5,0.25,1.207701
6,0.40,2.415401
7,1.00,inf


## Helpers

The family labels here split trigonal and hexagonal, because projection safety can differ between those two SG ranges.

In [5]:
def sg_family(sg: int | None) -> str | None:
    if sg is None:
        return None
    sg = int(sg)
    if sg <= 0:
        return None
    if sg <= 2:
        return "triclinic"
    if sg <= 15:
        return "monoclinic"
    if sg <= 74:
        return "orthorhombic"
    if sg <= 142:
        return "tetragonal"
    if sg <= 167:
        return "trigonal"
    if sg <= 194:
        return "hexagonal"
    if sg <= 230:
        return "cubic"
    return None


def get_structure_id(dataset: Any, i: int) -> str | None:
    try:
        return str(dataset.data.structure_id[i])
    except Exception:
        return None


def load_raw_sg_map(split: str) -> dict[str, int]:
    path = root / str(dataset_cfg["name"]) / "raw" / f"{split}.csv"
    if not path.exists():
        return {}
    mapping: dict[str, int] = {}
    with path.open("r", encoding="utf-8", newline="") as handle:
        reader = csv.DictReader(handle)
        for row in reader:
            material_id = row.get("material_id")
            value = row.get("spacegroup.number")
            if material_id and value not in {None, ""}:
                mapping[str(material_id)] = int(float(value))
    return mapping


def sample_tensors(sample: Any) -> dict[str, torch.Tensor | int]:
    return {
        "f": torch.as_tensor(sample.pos, device=DEVICE).reshape(-1, 3).float(),
        "l": torch.as_tensor(sample.l, device=DEVICE).reshape(1, 6).float(),
        "a": torch.as_tensor(sample.atomic_numbers, device=DEVICE).reshape(-1).long(),
        "sg": int(torch.as_tensor(sample.space_group).reshape(-1)[0].item()),
    }


def projection_stats(k: torch.Tensor, sg: int) -> dict[str, Any]:
    k = torch.as_tensor(k, device=DEVICE).reshape(1, 6).float()
    sg_t = torch.tensor([int(sg)], device=DEVICE, dtype=torch.long)
    k_proj = sym.proj_k_to_spacegroup(k, sg_t)
    delta = k - k_proj
    return {
        "k_proj": k_proj,
        "proj_abs_mean": float(delta.abs().mean().item()),
        "proj_l2": float(torch.linalg.norm(delta.reshape(-1)).item()),
        "proj_mse": float(delta.square().mean().item()),
        "proj_rel_l2": float((torch.linalg.norm(delta.reshape(-1)) / torch.linalg.norm(k.reshape(-1)).clamp_min(1e-12)).item()),
    }


def _float_or_nan(value: Any) -> float:
    if value is None:
        return float("nan")
    return float(value)


def eval_row(result: Any, requested_sg: int) -> dict[str, Any]:
    detected_sg = result.detected_space_group
    requested_family = sg_family(requested_sg)
    detected_family = sg_family(detected_sg)
    return {
        "valid": float(result.valid),
        "match": float(result.match),
        "rmse": _float_or_nan(result.rmse),
        "frac_rmse": _float_or_nan(result.frac_rmse),
        "frac_rmse_status": result.frac_rmse_status,
        "lengths_rmse": _float_or_nan(result.lattice_lengths_rmse),
        "angles_rmse": _float_or_nan(result.lattice_angles_rmse),
        "volume_rel_error": _float_or_nan(result.volume_rel_error),
        "composition_match": np.nan if result.composition_match is None else float(result.composition_match),
        "requested_sg_match": np.nan if result.requested_space_group_match is None else float(result.requested_space_group_match),
        "detected_sg_agreement": np.nan if result.requested_space_group_match is None else float(result.requested_space_group_match),
        "requested_family": requested_family,
        "detected_sg": detected_sg,
        "detected_family": detected_family,
        "detected_family_agreement": np.nan if requested_family is None or detected_family is None else float(requested_family == detected_family),
        "wyckoff_letter_agreement": np.nan if result.wyckoff_letter_agreement is None else float(result.wyckoff_letter_agreement),
        "predicted_wyckoff_dimensionality": json.dumps(result.predicted_wyckoff_dimensionality or {}, sort_keys=True),
        "target_wyckoff_dimensionality": json.dumps(result.target_wyckoff_dimensionality or {}, sort_keys=True),
        "validity_reason": result.validity_reason,
    }


def evaluate_pair(k_pred: torch.Tensor, t: dict[str, torch.Tensor | int]) -> Any:
    return evaluate_csp_reconstruction(
        pred_f=t["f"],
        pred_l=k_pred.reshape(1, 6),
        pred_a=t["a"],
        target_f=t["f"],
        target_l=t["l"],
        target_a=t["a"],
        lattice_transform=lattice_transform,
        requested_space_group=int(t["sg"]),
        stol=STOL,
        angle_tol=ANGLE_TOL,
        ltol=LTOL,
        sg_symprec=SG_SYMPREC,
        sg_angle_tolerance=SG_ANGLE_TOLERANCE,
        validity_cutoff=VALIDITY_CUTOFF,
    )


def p75(series: pd.Series) -> float:
    return float(series.dropna().quantile(0.75)) if series.notna().any() else np.nan


def p90(series: pd.Series) -> float:
    return float(series.dropna().quantile(0.90)) if series.notna().any() else np.nan


def p95(series: pd.Series) -> float:
    return float(series.dropna().quantile(0.95)) if series.notna().any() else np.nan


def p99(series: pd.Series) -> float:
    return float(series.dropna().quantile(0.99)) if series.notna().any() else np.nan

## Load the split

This uses the same transformed data path as KLDMPlus. The `l` field is already the DiffCSP++ `k` vector for this config.

In [6]:
raw_sg_map = load_raw_sg_map(SPLIT)
try:
    dataset = task.fit_dataset(root=root, split=SPLIT, download=DOWNLOAD_DATA)
except ImportError as exc:
    raise RuntimeError(
        "Dataset loading failed. This usually means the notebook is not running in the "
        "KLDMPlus/MatterGen environment; the current error was: " + str(exc)
    ) from exc

num_samples = len(dataset) if MAX_SAMPLES is None else min(len(dataset), int(MAX_SAMPLES))

samples: list[dict[str, Any]] = []
for idx in range(num_samples):
    sample = dataset[idx]
    sid = get_structure_id(dataset, idx)
    t = sample_tensors(sample)
    samples.append(
        {
            "idx": idx,
            "structure_id": sid,
            "sample": sample,
            "sg": int(t["sg"]),
            "raw_csv_sg": raw_sg_map.get(sid),
            "family": sg_family(int(t["sg"])),
        }
    )

print(f"loaded split={SPLIT} n={len(samples)} of total={len(dataset)}")
display(pd.DataFrame(samples).drop(columns=["sample"]).head())

dataset_cache action=load dataset=mp_20 split=train reason=fresh path=/workspace/data/mp_20/processed/train
dataset_cache action=from_cache_path:start dataset=mp_20 split=train
dataset_cache action=from_cache_path:done dataset=mp_20 split=train
dataset_cache action=builder_build:start dataset=mp_20 split=train
dataset_cache action=builder_build:done dataset=mp_20 split=train
loaded split=train n=512 of total=27136


,idx,structure_id,sg,raw_csv_sg,family
0,0,mp-1221227,8,None,monoclinic
1,1,mp-974729,139,None,tetragonal
2,2,mp-1185360,225,None,cubic
3,3,mp-1188861,62,None,orthorhombic
4,4,mp-677272,122,None,tetragonal


## Step 1: raw self-check

First verify that `(F0, k0, A)` evaluated against itself is basically perfect. If this fails, do not interpret the projection experiment yet; fix the evaluator or lattice transform first.

In [7]:
self_results = []
self_rows = []

for item in samples:
    t = sample_tensors(item["sample"])
    result = evaluate_pair(t["l"], t)
    self_results.append(result)
    self_rows.append(
        {
            "idx": item["idx"],
            "structure_id": item["structure_id"],
            "sg": item["sg"],
            "family": item["family"],
            **eval_row(result, int(t["sg"])),
        }
    )

self_df = pd.DataFrame(self_rows)
self_metrics = aggregate_csp_reconstruction_metrics(self_results)

display(pd.Series(self_metrics, name="raw_self_check"))
display(self_df.sort_values(["match", "valid", "rmse"], ascending=[True, True, False]).head(20))

self_df.to_csv(OUTPUT_DIR / f"{SPLIT}_raw_self_check.csv", index=False)

/workspace/.venv/lib/python3.11/site-packages/pymatgen/core/composition.py:373: UserWarning: No Pauling electronegativity for He. Setting to NaN. This has no physical meaning, and is mainly done to avoid errors caused by the code expecting a float.
  syms = sorted(sym_amt, key=lambda sym: get_el_sp(sym).X)


num_samples                                                                           512
valid                                                                                 1.0
match_rate                                                                            1.0
rmse                                                                                  0.0
rmse_defined_count                                                                    512
frac_rmse                                                                             0.0
frac_rmse_defined_count                                                               512
standardized_frac_rmse                                                               None
standardized_frac_rmse_defined_count                                                    0
composition_match_rate                                                                1.0
requested_space_group_match_rate                                                     None
detected_s

,idx,structure_id,sg,family,valid,match,rmse,frac_rmse,frac_rmse_status,lengths_rmse,...,requested_sg_match,detected_sg_agreement,requested_family,detected_sg,detected_family,detected_family_agreement,wyckoff_letter_agreement,predicted_wyckoff_dimensionality,target_wyckoff_dimensionality,validity_reason
74,74,mp-21437,136,tetragonal,1.0,1.0,8.982008e-06,0.0,ok,0.0,...,NaN,NaN,tetragonal,None,None,NaN,1.0,"{""0"": 1, ""1"": 2, ""2"": 1}","{""0"": 1, ""1"": 2, ""2"": 1}",ok
441,441,mp-1282563,4,monoclinic,1.0,1.0,7.801244e-06,0.0,ok,0.0,...,NaN,NaN,monoclinic,None,None,NaN,1.0,"{""3"": 6}","{""3"": 6}",ok
485,485,mp-862748,139,tetragonal,1.0,1.0,5.032010e-06,0.0,ok,0.0,...,NaN,NaN,tetragonal,None,None,NaN,1.0,"{""0"": 3, ""1"": 2}","{""0"": 3, ""1"": 2}",ok
91,91,mp-1104668,43,orthorhombic,1.0,1.0,2.044226e-06,0.0,ok,0.0,...,NaN,NaN,orthorhombic,None,None,NaN,1.0,"{""1"": 1, ""3"": 3}","{""1"": 1, ""3"": 3}",ok
329,329,mp-1217801,107,tetragonal,1.0,1.0,1.676101e-06,0.0,ok,0.0,...,NaN,NaN,tetragonal,None,None,NaN,1.0,"{""1"": 15}","{""1"": 15}",ok
45,45,mp-1224823,121,tetragonal,1.0,1.0,1.616482e-06,0.0,ok,0.0,...,NaN,NaN,tetragonal,None,None,NaN,1.0,"{""0"": 3, ""2"": 1}","{""0"": 3, ""2"": 1}",ok
309,309,mp-20790,82,tetragonal,1.0,1.0,1.447295e-06,0.0,ok,0.0,...,NaN,NaN,tetragonal,None,None,NaN,1.0,"{""0"": 2, ""3"": 1}","{""0"": 2, ""3"": 1}",ok
218,218,mp-559847,38,orthorhombic,1.0,1.0,9.837567e-07,0.0,ok,0.0,...,NaN,NaN,orthorhombic,None,None,NaN,1.0,"{""1"": 2, ""2"": 3, ""3"": 1}","{""1"": 2, ""2"": 3, ""3"": 1}",ok
82,82,mp-1219073,35,orthorhombic,1.0,1.0,8.973083e-07,0.0,ok,0.0,...,NaN,NaN,orthorhombic,None,None,NaN,1.0,"{""1"": 6, ""2"": 2}","{""1"": 6, ""2"": 2}",ok
272,272,mp-1223095,42,orthorhombic,1.0,1.0,1.478377e-07,0.0,ok,0.0,...,NaN,NaN,orthorhombic,None,None,NaN,1.0,"{""1"": 2, ""2"": 6, ""3"": 1}","{""1"": 2, ""2"": 6, ""3"": 1}",ok


## Step 2: ground-truth projection residual statistics

This is the current `proj_gt_k` question, but reported globally and by family with both absolute and relative scales.

In [8]:
proj_rows = []

for item in samples:
    t = sample_tensors(item["sample"])
    stats = projection_stats(t["l"], int(t["sg"]))
    proj_rows.append(
        {
            "idx": item["idx"],
            "structure_id": item["structure_id"],
            "sg": item["sg"],
            "raw_csv_sg": item["raw_csv_sg"],
            "family": item["family"],
            "proj_abs_mean": stats["proj_abs_mean"],
            "proj_l2": stats["proj_l2"],
            "proj_mse": stats["proj_mse"],
            "proj_rel_l2": stats["proj_rel_l2"],
        }
    )

proj_df = pd.DataFrame(proj_rows)

percentiles = [0.50, 0.75, 0.90, 0.95, 0.99]
display(proj_df[["proj_abs_mean", "proj_l2", "proj_mse", "proj_rel_l2"]].describe(percentiles=percentiles))

proj_by_family = (
    proj_df.groupby("family")
    .agg(
        n=("idx", "size"),
        mean_proj_abs=("proj_abs_mean", "mean"),
        median_proj_abs=("proj_abs_mean", "median"),
        p75_proj_abs=("proj_abs_mean", p75),
        p90_proj_abs=("proj_abs_mean", p90),
        p95_proj_abs=("proj_abs_mean", p95),
        p99_proj_abs=("proj_abs_mean", p99),
        max_proj_abs=("proj_abs_mean", "max"),
        mean_proj_rel=("proj_rel_l2", "mean"),
        p95_proj_rel=("proj_rel_l2", p95),
    )
    .reset_index()
    .sort_values("mean_proj_abs", ascending=False)
)

display(proj_by_family)
display(proj_df.sort_values("proj_abs_mean", ascending=False).head(25))

proj_df.to_csv(OUTPUT_DIR / f"{SPLIT}_projection_residuals.csv", index=False)
proj_by_family.to_csv(OUTPUT_DIR / f"{SPLIT}_projection_residuals_by_family.csv", index=False)

,proj_abs_mean,proj_l2,proj_mse,proj_rel_l2
count,512.000000,512.000000,512.000000,512.000000
mean,0.086478,0.334515,0.030920,0.115391
std,0.074464,0.271596,0.032957,0.096307
min,0.000000,0.000000,0.000000,0.000000
50%,0.077708,0.369684,0.022778,0.122393
75%,0.163376,0.565952,0.053384,0.202051
90%,0.164518,0.626340,0.065384,0.235980
95%,0.204869,0.806123,0.108306,0.270819
99%,0.226679,0.846799,0.119511,0.309143
max,0.248707,0.901967,0.135591,0.339462


,family,n,mean_proj_abs,median_proj_abs,p75_proj_abs,p90_proj_abs,p95_proj_abs,p99_proj_abs,max_proj_abs,mean_proj_rel,p95_proj_rel
0,cubic,121,0.136372,0.163376,0.163376,0.163377,0.163377,0.163377,0.163377,0.174305,0.244013
6,trigonal,66,0.129843,0.159761,0.215934,0.226230,0.226679,0.227311,0.228013,0.180398,0.310655
1,hexagonal,50,0.086108,0.132878,0.153575,0.184369,0.206105,0.238421,0.248707,0.106291,0.247537
4,tetragonal,78,0.080701,0.080914,0.136416,0.163478,0.163971,0.176325,0.177208,0.103744,0.210170
3,orthorhombic,93,0.052359,0.038310,0.096003,0.143208,0.162230,0.166605,0.166809,0.072689,0.184564
2,monoclinic,85,0.043957,0.041486,0.064795,0.093711,0.097963,0.109212,0.110648,0.069605,0.143373
5,triclinic,19,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


,idx,structure_id,sg,raw_csv_sg,family,proj_abs_mean,proj_l2,proj_mse,proj_rel_l2
227,227,mp-864657,194,None,hexagonal,0.248707,0.901967,0.135591,0.245431
78,78,mp-1221909,166,None,trigonal,0.228013,0.851281,0.120780,0.260542
6,6,mp-561310,189,None,hexagonal,0.227715,0.805725,0.108199,0.253202
200,200,mp-755568,161,None,trigonal,0.226934,0.849240,0.120201,0.298614
470,470,mp-1215517,166,None,trigonal,0.226921,0.849216,0.120195,0.320699
281,281,mp-1247173,166,None,trigonal,0.226680,0.846835,0.119522,0.257721
186,186,mp-1217066,166,None,trigonal,0.226677,0.848758,0.120065,0.334423
86,86,mp-989588,167,None,trigonal,0.226518,0.846346,0.119384,0.298591
76,76,mp-1217696,166,None,trigonal,0.226514,0.846335,0.119381,0.311114
170,170,mp-1540553,166,None,trigonal,0.225946,0.844510,0.118866,0.257497


## Step 3: soft interpolation safety test

For each `rho`, this evaluates:

```text
(F0, k_rho, A) vs (F0, k0, A)
```

where fractional coordinates are deliberately left untouched. This is the relevant oracle for direct SG loss safety.

In [9]:
results_by_rho: dict[float, list[Any]] = {float(rho): [] for rho in RHOS}
rows: list[dict[str, Any]] = []

for n_done, item in enumerate(samples, start=1):
    t = sample_tensors(item["sample"])
    k0 = t["l"].reshape(1, 6)
    sg = int(t["sg"])
    raw_stats = projection_stats(k0, sg)
    k_proj = raw_stats["k_proj"].reshape(1, 6)
    raw_abs = float(raw_stats["proj_abs_mean"])
    raw_l2 = float(raw_stats["proj_l2"])

    for rho in RHOS:
        rho = float(rho)
        k_rho = (1.0 - rho) * k0 + rho * k_proj
        after_stats = projection_stats(k_rho, sg)
        move_delta = k_rho - k0
        move_l2 = float(torch.linalg.norm(move_delta.reshape(-1)).item())
        move_abs = float(move_delta.abs().mean().item())

        result = evaluate_pair(k_rho, t)
        results_by_rho[rho].append(result)

        rows.append(
            {
                "idx": item["idx"],
                "structure_id": item["structure_id"],
                "sg": item["sg"],
                "raw_csv_sg": item["raw_csv_sg"],
                "family": item["family"],
                "rho": rho,
                "lambda_equiv_at_mean_weight": rho_to_lambda(rho, mean_time_weight),
                "proj_abs_mean_raw": raw_abs,
                "proj_l2_raw": raw_l2,
                "proj_rel_l2_raw": float(raw_stats["proj_rel_l2"]),
                "proj_abs_mean_after": after_stats["proj_abs_mean"],
                "proj_l2_after": after_stats["proj_l2"],
                "proj_rel_l2_after": after_stats["proj_rel_l2"],
                "proj_reduction_abs_mean": np.nan if raw_abs <= 1e-12 else 1.0 - after_stats["proj_abs_mean"] / raw_abs,
                "proj_reduction_l2": np.nan if raw_l2 <= 1e-12 else 1.0 - after_stats["proj_l2"] / raw_l2,
                "k_move_abs_mean": move_abs,
                "k_move_l2": move_l2,
                "k_move_rel_l2": np.nan if torch.linalg.norm(k0.reshape(-1)).item() <= 1e-12 else move_l2 / float(torch.linalg.norm(k0.reshape(-1)).item()),
                **eval_row(result, sg),
            }
        )

    if n_done % 50 == 0:
        print(f"evaluated {n_done}/{len(samples)} samples")

df = pd.DataFrame(rows)
display(df.head())

df.to_csv(OUTPUT_DIR / f"{SPLIT}_soft_projection_safety_per_sample.csv", index=False)
print("saved", OUTPUT_DIR / f"{SPLIT}_soft_projection_safety_per_sample.csv")

evaluated 50/512 samples
evaluated 100/512 samples
evaluated 150/512 samples
evaluated 200/512 samples


/workspace/.venv/lib/python3.11/site-packages/pymatgen/core/composition.py:373: UserWarning: No Pauling electronegativity for He. Setting to NaN. This has no physical meaning, and is mainly done to avoid errors caused by the code expecting a float.
  syms = sorted(sym_amt, key=lambda sym: get_el_sp(sym).X)
/workspace/.venv/lib/python3.11/site-packages/pymatgen/core/composition.py:373: UserWarning: No Pauling electronegativity for He. Setting to NaN. This has no physical meaning, and is mainly done to avoid errors caused by the code expecting a float.
  syms = sorted(sym_amt, key=lambda sym: get_el_sp(sym).X)
/workspace/.venv/lib/python3.11/site-packages/pymatgen/core/composition.py:373: UserWarning: No Pauling electronegativity for He. Setting to NaN. This has no physical meaning, and is mainly done to avoid errors caused by the code expecting a float.
  syms = sorted(sym_amt, key=lambda sym: get_el_sp(sym).X)
/workspace/.venv/lib/python3.11/site-packages/pymatgen/core/composition.py:3

evaluated 250/512 samples
evaluated 300/512 samples
evaluated 350/512 samples
evaluated 400/512 samples
evaluated 450/512 samples
evaluated 500/512 samples


,idx,structure_id,sg,raw_csv_sg,family,rho,lambda_equiv_at_mean_weight,proj_abs_mean_raw,proj_l2_raw,proj_rel_l2_raw,...,requested_sg_match,detected_sg_agreement,requested_family,detected_sg,detected_family,detected_family_agreement,wyckoff_letter_agreement,predicted_wyckoff_dimensionality,target_wyckoff_dimensionality,validity_reason
0,0,mp-1221227,8,None,monoclinic,0.00,0.000000,0.040951,0.218798,0.075649,...,NaN,NaN,monoclinic,None,None,NaN,1.0,"{""2"": 12}","{""2"": 12}",ok
1,0,mp-1221227,8,None,monoclinic,0.03,0.112055,0.040951,0.218798,0.075649,...,NaN,NaN,monoclinic,None,None,NaN,1.0,"{""2"": 12}","{""2"": 12}",ok
2,0,mp-1221227,8,None,monoclinic,0.05,0.190690,0.040951,0.218798,0.075649,...,NaN,NaN,monoclinic,None,None,NaN,1.0,"{""2"": 12}","{""2"": 12}",ok
3,0,mp-1221227,8,None,monoclinic,0.09,0.358329,0.040951,0.218798,0.075649,...,NaN,NaN,monoclinic,None,None,NaN,1.0,"{""2"": 12}","{""2"": 12}",ok
4,0,mp-1221227,8,None,monoclinic,0.15,0.639371,0.040951,0.218798,0.075649,...,NaN,NaN,monoclinic,None,None,NaN,1.0,"{""2"": 12}","{""2"": 12}",ok


saved /workspace/notebooks/outputs/oracle_projection_safety/train_soft_projection_safety_per_sample.csv


## Step 4: aggregate by rho

This table is the main safety curve. A safe `rho` should preserve match rate and keep lattice/volume damage near zero while still reducing projection residual.

In [10]:
def metric_value(value: Any) -> Any:
    if isinstance(value, dict):
        return json.dumps(value, sort_keys=True)
    return value

summary_rows = []
for rho in RHOS:
    rho = float(rho)
    rho_df = df[df["rho"] == rho]
    metrics = aggregate_csp_reconstruction_metrics(results_by_rho[rho])
    summary_rows.append(
        {
            "rho": rho,
            "lambda_equiv_at_mean_weight": rho_to_lambda(rho, mean_time_weight),
            "n": int(len(rho_df)),
            "mean_proj_abs_raw": float(rho_df["proj_abs_mean_raw"].mean()),
            "mean_proj_abs_after": float(rho_df["proj_abs_mean_after"].mean()),
            "mean_proj_reduction_abs": float(rho_df["proj_reduction_abs_mean"].mean()),
            "mean_k_move_abs": float(rho_df["k_move_abs_mean"].mean()),
            "p95_k_move_abs": p95(rho_df["k_move_abs_mean"]),
            "p95_volume_rel_error": p95(rho_df["volume_rel_error"]),
            "p95_angles_rmse": p95(rho_df["angles_rmse"]),
            "p95_lengths_rmse": p95(rho_df["lengths_rmse"]),
            **{key: metric_value(value) for key, value in metrics.items()},
        }
    )

summary_df = pd.DataFrame(summary_rows)
display(summary_df)
summary_df.to_csv(OUTPUT_DIR / f"{SPLIT}_soft_projection_safety_by_rho.csv", index=False)

,rho,lambda_equiv_at_mean_weight,n,mean_proj_abs_raw,mean_proj_abs_after,mean_proj_reduction_abs,mean_k_move_abs,p95_k_move_abs,p95_volume_rel_error,p95_angles_rmse,...,requested_space_group_match_rate,detected_sg_agreement,detected_family_agreement,wyckoff_letter_agreement,predicted_wyckoff_dimensionality_distribution,target_wyckoff_dimensionality_distribution,lattice_lengths_rmse,lattice_angles_rmse,volume_rel_error,matcher_diagnosis_counts
0,0.00,0.000000,512,0.086478,0.086478,0.000000,0.000000,0.000000,0.000000e+00,0.000000,...,None,None,None,1.000000,"{""0"": 771, ""1"": 616, ""2"": 459, ""3"": 592}","{""0"": 771, ""1"": 616, ""2"": 459, ""3"": 592}",0.000000,0.000000,0.000000e+00,{}
1,0.03,0.112055,512,0.086478,0.083883,0.028695,0.002594,0.006146,6.134153e-07,1.089577,...,None,None,None,0.556641,"{""0"": 616, ""1"": 456, ""2"": 653, ""3"": 1073}","{""0"": 771, ""1"": 616, ""2"": 459, ""3"": 592}",0.018889,0.453875,2.037689e-07,{}
2,0.05,0.190690,512,0.086478,0.082154,0.020153,0.004324,0.010243,4.499881e-07,1.823330,...,None,None,None,0.542969,"{""0"": 618, ""1"": 433, ""2"": 661, ""3"": 1118}","{""0"": 771, ""1"": 616, ""2"": 459, ""3"": 592}",0.031120,0.757927,1.411864e-07,{}
3,0.09,0.358329,512,0.086478,0.078695,0.056039,0.007783,0.018438,6.273327e-07,3.307534,...,None,None,None,0.523438,"{""0"": 621, ""1"": 397, ""2"": 526, ""3"": 1350}","{""0"": 771, ""1"": 616, ""2"": 459, ""3"": 592}",0.054715,1.369351,2.065946e-07,{}
4,0.15,0.639371,512,0.086478,0.073506,0.148118,0.012972,0.030730,6.542915e-07,5.571920,...,None,None,None,0.519531,"{""0"": 624, ""1"": 395, ""2"": 522, ""3"": 1359}","{""0"": 771, ""1"": 616, ""2"": 459, ""3"": 592}",0.087957,2.294072,2.317265e-07,{}
5,0.25,1.207701,512,0.086478,0.064858,0.252328,0.021619,0.051217,3.537385e-07,9.428785,...,None,None,None,0.517578,"{""0"": 626, ""1"": 392, ""2"": 497, ""3"": 1392}","{""0"": 771, ""1"": 616, ""2"": 459, ""3"": 592}",0.137687,3.852026,1.203879e-07,"{""different_motif_after_standardization"": 4}"
6,0.40,2.415401,512,0.086478,0.051887,0.400091,0.034591,0.081947,7.009198e-07,15.328542,...,None,None,None,0.515625,"{""0"": 627, ""1"": 392, ""2"": 496, ""3"": 1393}","{""0"": 771, ""1"": 616, ""2"": 459, ""3"": 592}",0.199275,6.214120,2.329791e-07,"{""different_motif_after_standardization"": 44}"
7,1.00,inf,512,0.086478,0.000000,1.000000,0.086478,0.204869,3.809150e-07,37.358312,...,None,None,None,0.509766,"{""0"": 625, ""1"": 392, ""2"": 483, ""3"": 1405}","{""0"": 771, ""1"": 616, ""2"": 459, ""3"": 592}",0.307260,15.456502,1.331793e-07,"{""different_motif_after_standardization"": 312}"


## Step 5: aggregate by rho and family

Use this to decide whether direct SG guidance is only safe for some families. For example, a method may be harmless for low-symmetry families but damaging for cubic or hexagonal/trigonal cells.

In [11]:
family_summary = (
    df.groupby(["rho", "family"])
    .agg(
        n=("idx", "size"),
        match_rate=("match", "mean"),
        valid=("valid", "mean"),
        rmse_mean=("rmse", "mean"),
        rmse_p95=("rmse", p95),
        frac_rmse_mean=("frac_rmse", "mean"),
        lengths_rmse_mean=("lengths_rmse", "mean"),
        lengths_rmse_p95=("lengths_rmse", p95),
        angles_rmse_mean=("angles_rmse", "mean"),
        angles_rmse_p95=("angles_rmse", p95),
        volume_rel_error_mean=("volume_rel_error", "mean"),
        volume_rel_error_p95=("volume_rel_error", p95),
        detected_family_agreement=("detected_family_agreement", "mean"),
        detected_sg_agreement=("detected_sg_agreement", "mean"),
        wyckoff_letter_agreement=("wyckoff_letter_agreement", "mean"),
        proj_abs_raw=("proj_abs_mean_raw", "mean"),
        proj_abs_after=("proj_abs_mean_after", "mean"),
        proj_reduction_abs=("proj_reduction_abs_mean", "mean"),
        k_move_abs_mean=("k_move_abs_mean", "mean"),
        k_move_abs_p95=("k_move_abs_mean", p95),
    )
    .reset_index()
    .sort_values(["rho", "match_rate", "volume_rel_error_mean"], ascending=[True, True, False])
)

display(family_summary)
family_summary.to_csv(OUTPUT_DIR / f"{SPLIT}_soft_projection_safety_by_rho_family.csv", index=False)

,rho,family,n,match_rate,valid,rmse_mean,rmse_p95,frac_rmse_mean,lengths_rmse_mean,lengths_rmse_p95,...,volume_rel_error_mean,volume_rel_error_p95,detected_family_agreement,detected_sg_agreement,wyckoff_letter_agreement,proj_abs_raw,proj_abs_after,proj_reduction_abs,k_move_abs_mean,k_move_abs_p95
0,0.00,cubic,121,1.000000,1.0,1.734563e-09,4.359060e-16,0.0,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,NaN,NaN,1.000000,0.136372,0.136372,0.000000,0.000000e+00,0.000000e+00
1,0.00,hexagonal,50,1.000000,1.0,1.796823e-16,3.150294e-16,0.0,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,NaN,NaN,1.000000,0.086108,0.086108,0.000000,0.000000e+00,0.000000e+00
2,0.00,monoclinic,85,1.000000,1.0,1.360382e-07,9.929805e-08,0.0,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,NaN,NaN,1.000000,0.043957,0.043957,0.000000,0.000000e+00,0.000000e+00
3,0.00,orthorhombic,93,1.000000,1.0,7.147793e-08,1.312127e-07,0.0,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,NaN,NaN,1.000000,0.052359,0.052359,0.000000,0.000000e+00,0.000000e+00
4,0.00,tetragonal,78,1.000000,1.0,2.658589e-07,1.472673e-06,0.0,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,NaN,NaN,1.000000,0.080701,0.080701,0.000000,0.000000e+00,0.000000e+00
5,0.00,triclinic,19,1.000000,1.0,3.949739e-08,9.049276e-08,0.0,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,NaN,NaN,1.000000,0.000000,0.000000,NaN,0.000000e+00,0.000000e+00
6,0.00,trigonal,66,1.000000,1.0,1.789310e-09,4.184105e-16,0.0,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,NaN,NaN,1.000000,0.129843,0.129843,0.000000,0.000000e+00,0.000000e+00
7,0.03,cubic,121,1.000000,1.0,1.734563e-09,5.955582e-16,0.0,2.982372e-02,5.048113e-02,...,2.558682e-07,6.286601e-07,NaN,NaN,0.636364,0.136372,0.132281,0.030000,4.091177e-03,4.901331e-03
13,0.03,trigonal,66,1.000000,1.0,3.172134e-09,1.642680e-08,0.0,2.727291e-02,6.364177e-02,...,2.165357e-07,6.040118e-07,NaN,NaN,0.393939,0.129843,0.125948,0.025538,3.895295e-03,6.800361e-03
11,0.03,tetragonal,78,1.000000,1.0,2.659374e-07,1.477315e-06,0.0,1.861088e-02,4.954490e-02,...,2.116215e-07,6.467506e-07,NaN,NaN,0.307692,0.080701,0.078280,0.030000,2.421030e-03,4.919125e-03


## Step 6: worst-case inspection

These tables show the samples that projection damages most. If failures cluster in one family or a few pathological examples, family-specific or robust weighting may be enough. If failures are broad even at small `rho`, direct SG loss is probably unsafe in this chart.

In [12]:
for rho in RHOS:
    rho = float(rho)
    rho_df = df[df["rho"] == rho].copy()
    print("rho", rho)
    display(
        rho_df.sort_values(
            ["match", "volume_rel_error", "angles_rmse", "lengths_rmse"],
            ascending=[True, False, False, False],
        )[
            [
                "idx",
                "structure_id",
                "sg",
                "family",
                "proj_abs_mean_raw",
                "proj_abs_mean_after",
                "proj_reduction_abs_mean",
                "k_move_abs_mean",
                "valid",
                "match",
                "rmse",
                "frac_rmse",
                "lengths_rmse",
                "angles_rmse",
                "volume_rel_error",
                "detected_sg",
                "detected_family",
                "detected_family_agreement",
                "detected_sg_agreement",
                "wyckoff_letter_agreement",
                "validity_reason",
            ]
        ].head(15)
    )

rho 0.0


,idx,structure_id,sg,family,proj_abs_mean_raw,proj_abs_mean_after,proj_reduction_abs_mean,k_move_abs_mean,valid,match,...,frac_rmse,lengths_rmse,angles_rmse,volume_rel_error,detected_sg,detected_family,detected_family_agreement,detected_sg_agreement,wyckoff_letter_agreement,validity_reason
0,0,mp-1221227,8,monoclinic,4.095083e-02,4.095083e-02,0.0,0.0,1.0,1.0,...,0.0,0.0,0.0,0.0,None,None,NaN,NaN,1.0,ok
8,1,mp-974729,139,tetragonal,1.397276e-01,1.397276e-01,0.0,0.0,1.0,1.0,...,0.0,0.0,0.0,0.0,None,None,NaN,NaN,1.0,ok
16,2,mp-1185360,225,cubic,1.633763e-01,1.633763e-01,0.0,0.0,1.0,1.0,...,0.0,0.0,0.0,0.0,None,None,NaN,NaN,1.0,ok
24,3,mp-1188861,62,orthorhombic,0.000000e+00,0.000000e+00,NaN,0.0,1.0,1.0,...,0.0,0.0,0.0,0.0,None,None,NaN,NaN,1.0,ok
32,4,mp-677272,122,tetragonal,1.636317e-01,1.636317e-01,0.0,0.0,1.0,1.0,...,0.0,0.0,0.0,0.0,None,None,NaN,NaN,1.0,ok
40,5,mp-1104517,71,orthorhombic,8.496859e-02,8.496859e-02,0.0,0.0,1.0,1.0,...,0.0,0.0,0.0,0.0,None,None,NaN,NaN,1.0,ok
48,6,mp-561310,189,hexagonal,2.277149e-01,2.277149e-01,0.0,0.0,1.0,1.0,...,0.0,0.0,0.0,0.0,None,None,NaN,NaN,1.0,ok
56,7,mp-777964,12,monoclinic,9.756353e-02,9.756353e-02,0.0,0.0,1.0,1.0,...,0.0,0.0,0.0,0.0,None,None,NaN,NaN,1.0,ok
64,8,mp-1078776,11,monoclinic,2.671093e-03,2.671093e-03,0.0,0.0,1.0,1.0,...,0.0,0.0,0.0,0.0,None,None,NaN,NaN,1.0,ok
72,9,mp-1217581,65,orthorhombic,6.082542e-02,6.082542e-02,0.0,0.0,1.0,1.0,...,0.0,0.0,0.0,0.0,None,None,NaN,NaN,1.0,ok


rho 0.03


,idx,structure_id,sg,family,proj_abs_mean_raw,proj_abs_mean_after,proj_reduction_abs_mean,k_move_abs_mean,valid,match,...,frac_rmse,lengths_rmse,angles_rmse,volume_rel_error,detected_sg,detected_family,detected_family_agreement,detected_sg_agreement,wyckoff_letter_agreement,validity_reason
3249,406,mp-1205711,194,hexagonal,2.577430e-09,2.500107e-09,0.030000,3.981375e-08,1.0,1.0,...,0.0,5.558909e-06,0.000003,1.017401e-06,None,None,NaN,NaN,1.0,ok
2777,347,mp-29644,164,trigonal,2.459605e-08,2.430520e-08,0.011825,5.989549e-08,1.0,1.0,...,0.0,7.747552e-06,0.000002,1.007127e-06,None,None,NaN,NaN,1.0,ok
2785,348,mp-971739,225,cubic,1.633763e-01,1.584751e-01,0.030000,4.901326e-03,1.0,1.0,...,0.0,2.935742e-02,0.919217,9.961662e-07,None,None,NaN,NaN,1.0,ok
1025,128,mp-1205575,225,cubic,1.633765e-01,1.584752e-01,0.030000,4.901330e-03,1.0,1.0,...,0.0,3.898876e-02,0.919216,9.759251e-07,None,None,NaN,NaN,0.0,ok
65,8,mp-1078776,11,monoclinic,2.671093e-03,2.590960e-03,0.030000,8.017244e-05,1.0,1.0,...,0.0,4.885495e-05,0.023089,9.313371e-07,None,None,NaN,NaN,1.0,ok
329,41,mp-1225810,44,orthorhombic,9.860589e-02,9.564772e-02,0.030000,2.958221e-03,1.0,1.0,...,0.0,1.781386e-02,0.470060,8.372150e-07,None,None,NaN,NaN,0.0,ok
1401,175,mp-1030504,156,trigonal,3.387626e-08,3.315799e-08,0.021203,4.045469e-08,1.0,1.0,...,0.0,6.648526e-06,0.000003,7.982840e-07,None,None,NaN,NaN,1.0,ok
2905,363,mp-569548,139,tetragonal,1.610560e-01,1.562244e-01,0.030000,4.831718e-03,1.0,1.0,...,0.0,4.884068e-02,0.581633,7.807069e-07,None,None,NaN,NaN,0.0,ok
3601,450,mp-1018647,221,cubic,3.317495e-08,3.217971e-08,0.030000,4.073168e-08,1.0,1.0,...,0.0,8.813967e-07,0.000000,7.556954e-07,None,None,NaN,NaN,1.0,ok
1873,234,mp-867270,225,cubic,1.633764e-01,1.584751e-01,0.030000,4.901331e-03,1.0,1.0,...,0.0,2.626275e-02,0.919223,7.478868e-07,None,None,NaN,NaN,1.0,ok


rho 0.05


...

,idx,structure_id,sg,family,proj_abs_mean_raw,proj_abs_mean_after,proj_reduction_abs_mean,k_move_abs_mean,valid,match,...,frac_rmse,lengths_rmse,angles_rmse,volume_rel_error,detected_sg,detected_family,detected_family_agreement,detected_sg_agreement,wyckoff_letter_agreement,validity_reason
2773,346,mp-1216123,166,trigonal,0.222720,0.167040,0.25,0.055680,1.0,0.0,...,0.0,0.325826,11.092003,1.552428e-07,None,None,NaN,NaN,0.0,ok
613,76,mp-1217696,166,trigonal,0.226514,0.169886,0.25,0.056629,1.0,0.0,...,0.0,0.326586,11.250239,6.574894e-08,None,None,NaN,NaN,0.0,ok
2253,281,mp-1247173,166,trigonal,0.226680,0.170010,0.25,0.056670,1.0,0.0,...,0.0,0.456206,11.256828,3.605982e-08,None,None,NaN,NaN,0.0,ok
1365,170,mp-1540553,166,trigonal,0.225946,0.169459,0.25,0.056486,1.0,0.0,...,0.0,0.450683,11.226674,1.111725e-08,None,None,NaN,NaN,0.0,ok
125,15,mp-20950,139,tetragonal,0.104474,0.078356,0.25,0.026119,1.0,1.0,...,0.0,0.109770,4.082004,6.495801e-07,None,None,NaN,NaN,0.0,ok
1077,134,mp-1186253,225,cubic,0.163377,0.122532,0.25,0.040844,1.0,1.0,...,0.0,0.271223,7.748091,6.403127e-07,None,None,NaN,NaN,1.0,ok
3853,481,mp-865422,225,cubic,0.163376,0.122532,0.25,0.040844,1.0,1.0,...,0.0,0.235267,7.748080,5.697054e-07,None,None,NaN,NaN,1.0,ok
3557,444,mp-1078832,189,hexagonal,0.200131,0.150099,0.25,0.050033,1.0,1.0,...,0.0,0.433549,6.262769,5.183452e-07,None,None,NaN,NaN,0.0,ok
2725,340,mp-1114295,225,cubic,0.163376,0.122532,0.25,0.040844,1.0,1.0,...,0.0,0.388314,7.748079,5.153816e-07,None,None,NaN,NaN,0.0,ok
117,14,mp-1112148,225,cubic,0.163376,0.122532,0.25,0.040844,1.0,1.0,...,0.0,0.422044,7.748088,5.019894e-07,None,None,NaN,NaN,0.0,ok


rho 0.4


,idx,structure_id,sg,family,proj_abs_mean_raw,proj_abs_mean_after,proj_reduction_abs_mean,k_move_abs_mean,valid,match,...,frac_rmse,lengths_rmse,angles_rmse,volume_rel_error,detected_sg,detected_family,detected_family_agreement,detected_sg_agreement,wyckoff_letter_agreement,validity_reason
278,34,mp-15634,166,trigonal,0.181619,0.108972,0.4,0.072648,1.0,0.0,...,0.0,0.410112,15.254927,7.802375e-07,None,None,NaN,NaN,0.0,ok
2150,268,mp-1103015,166,trigonal,0.183481,0.110089,0.4,0.073392,1.0,0.0,...,0.0,0.328040,15.352119,6.349523e-07,None,None,NaN,NaN,0.0,ok
614,76,mp-1217696,166,trigonal,0.226514,0.135909,0.4,0.090606,1.0,0.0,...,0.0,0.437304,18.106976,5.702593e-07,None,None,NaN,NaN,0.0,ok
1070,133,mp-3827,15,monoclinic,0.093922,0.056353,0.4,0.037569,1.0,0.0,...,0.0,0.224845,6.579328,5.103145e-07,None,None,NaN,NaN,0.0,ok
3590,448,mp-19147,167,trigonal,0.208291,0.124975,0.4,0.083316,1.0,0.0,...,0.0,0.348728,16.855523,4.689350e-07,None,None,NaN,NaN,0.0,ok
3390,423,mp-1087504,166,trigonal,0.157606,0.094563,0.4,0.063042,1.0,0.0,...,0.0,0.235362,14.247144,4.623413e-07,None,None,NaN,NaN,0.0,ok
2750,343,mp-1219212,160,trigonal,0.109784,0.065870,0.4,0.043913,1.0,0.0,...,0.0,0.023609,9.561076,4.374314e-07,None,None,NaN,NaN,0.0,ok
2718,339,mp-1220689,166,trigonal,0.224709,0.134825,0.4,0.089884,1.0,0.0,...,0.0,0.457715,17.983381,4.361471e-07,None,None,NaN,NaN,0.0,ok
430,53,mp-676315,166,trigonal,0.161798,0.097079,0.4,0.064719,1.0,0.0,...,0.0,0.251106,14.388739,3.976122e-07,None,None,NaN,NaN,0.0,ok
1286,160,mp-9274,166,trigonal,0.205924,0.123555,0.4,0.082370,1.0,0.0,...,0.0,0.491344,16.698580,3.482124e-07,None,None,NaN,NaN,0.0,ok


rho 1.0


,idx,structure_id,sg,family,proj_abs_mean_raw,proj_abs_mean_after,proj_reduction_abs_mean,k_move_abs_mean,valid,match,...,frac_rmse,lengths_rmse,angles_rmse,volume_rel_error,detected_sg,detected_family,detected_family_agreement,detected_sg_agreement,wyckoff_letter_agreement,validity_reason
2175,271,mp-2694,227,cubic,0.163376,0.0,1.0,0.163376,1.0,0.0,...,0.0,0.628217,29.999975,7.047264e-07,None,None,NaN,NaN,0.0,ok
3839,479,mp-1209696,204,cubic,0.163376,0.0,1.0,0.163376,1.0,0.0,...,0.0,0.713772,19.471214,6.995438e-07,None,None,NaN,NaN,0.0,ok
3703,462,mp-1185337,139,tetragonal,0.131829,0.0,1.0,0.131829,1.0,0.0,...,0.0,0.384039,20.754022,6.462634e-07,None,None,NaN,NaN,0.0,ok
1095,136,mp-1184112,225,cubic,0.163377,0.0,1.0,0.163377,1.0,0.0,...,0.0,0.540819,30.000018,5.870474e-07,None,None,NaN,NaN,1.0,ok
151,18,mp-16341,225,cubic,0.163376,0.0,1.0,0.163376,1.0,0.0,...,0.0,0.499465,30.000006,5.385231e-07,None,None,NaN,NaN,1.0,ok
2791,348,mp-971739,225,cubic,0.163376,0.0,1.0,0.163376,1.0,0.0,...,0.0,0.470236,29.999987,5.284799e-07,None,None,NaN,NaN,1.0,ok
679,84,mp-675739,82,tetragonal,0.164591,0.0,1.0,0.164591,1.0,0.0,...,0.0,0.643480,19.525131,5.168199e-07,None,None,NaN,NaN,0.0,ok
1231,153,mp-1185443,225,cubic,0.163376,0.0,1.0,0.163376,1.0,0.0,...,0.0,0.580628,29.999985,4.889061e-07,None,None,NaN,NaN,1.0,ok
2407,300,mp-867201,225,cubic,0.163376,0.0,1.0,0.163376,1.0,0.0,...,0.0,0.572713,29.999999,4.781127e-07,None,None,NaN,NaN,1.0,ok
1519,189,mp-864769,225,cubic,0.163376,0.0,1.0,0.163376,1.0,0.0,...,0.0,0.593490,29.999999,4.664951e-07,None,None,NaN,NaN,1.0,ok


## Optional pass/fail helper

This is intentionally not a law of nature. It is a practical guardrail for deciding whether a proposed direct SG strength is safe enough to train with.

Suggested read:

- If `rho = 1.0` is bad but `rho = 0.03` or `0.09` is clean, hard projection is unsafe but small soft guidance is plausible.
- If `rho = 0.09` already hurts match rate, volume, or angles, direct SG regularization is unsafe in this data chart.
- If only a few families fail, consider family-specific SG loss or switch those families to orbit/canonicalization diagnostics.

In [13]:
PASS_RULES = {
    "min_global_match_rate": 0.98,
    "min_family_match_rate": 0.95,
    "max_mean_volume_rel_error": 0.02,
    "max_p95_volume_rel_error": 0.05,
    "max_mean_angles_rmse": 2.0,
    "max_p95_angles_rmse": 5.0,
}

rho_decision_rows = []
for rho in RHOS:
    rho = float(rho)
    global_row = summary_df[summary_df["rho"] == rho].iloc[0]
    fam = family_summary[family_summary["rho"] == rho]
    min_family_match = float(fam["match_rate"].min()) if len(fam) else np.nan

    passes = bool(
        global_row["match_rate"] >= PASS_RULES["min_global_match_rate"]
        and min_family_match >= PASS_RULES["min_family_match_rate"]
        and global_row["volume_rel_error"] <= PASS_RULES["max_mean_volume_rel_error"]
        and global_row["p95_volume_rel_error"] <= PASS_RULES["max_p95_volume_rel_error"]
        and global_row["lattice_angles_rmse"] <= PASS_RULES["max_mean_angles_rmse"]
        and global_row["p95_angles_rmse"] <= PASS_RULES["max_p95_angles_rmse"]
    )
    rho_decision_rows.append(
        {
            "rho": rho,
            "lambda_equiv_at_mean_weight": rho_to_lambda(rho, mean_time_weight),
            "passes_suggested_guardrail": passes,
            "global_match_rate": global_row["match_rate"],
            "min_family_match_rate": min_family_match,
            "mean_volume_rel_error": global_row["volume_rel_error"],
            "p95_volume_rel_error": global_row["p95_volume_rel_error"],
            "mean_angles_rmse": global_row["lattice_angles_rmse"],
            "p95_angles_rmse": global_row["p95_angles_rmse"],
            "mean_proj_reduction_abs": global_row["mean_proj_reduction_abs"],
        }
    )

decision_df = pd.DataFrame(rho_decision_rows)
display(pd.Series(PASS_RULES, name="suggested_guardrail"))
display(decision_df)
decision_df.to_csv(OUTPUT_DIR / f"{SPLIT}_soft_projection_safety_decision.csv", index=False)

min_global_match_rate        0.98
min_family_match_rate        0.95
max_mean_volume_rel_error    0.02
max_p95_volume_rel_error     0.05
max_mean_angles_rmse         2.00
max_p95_angles_rmse          5.00
Name: suggested_guardrail, dtype: float64

,rho,lambda_equiv_at_mean_weight,passes_suggested_guardrail,global_match_rate,min_family_match_rate,mean_volume_rel_error,p95_volume_rel_error,mean_angles_rmse,p95_angles_rmse,mean_proj_reduction_abs
0,0.00,0.000000,True,1.000000,1.000000,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000
1,0.03,0.112055,True,1.000000,1.000000,2.037689e-07,6.134153e-07,0.453875,1.089577,0.028695
2,0.05,0.190690,True,1.000000,1.000000,1.411864e-07,4.499881e-07,0.757927,1.823330,0.020153
3,0.09,0.358329,True,1.000000,1.000000,2.065946e-07,6.273327e-07,1.369351,3.307534,0.056039
4,0.15,0.639371,False,1.000000,1.000000,2.317265e-07,6.542915e-07,2.294072,5.571920,0.148118
5,0.25,1.207701,False,0.992188,0.939394,1.203879e-07,3.537385e-07,3.852026,9.428785,0.252328
6,0.40,2.415401,False,0.914062,0.424242,2.329791e-07,7.009198e-07,6.214120,15.328542,0.400091
7,1.00,inf,False,0.390625,0.165289,1.331793e-07,3.809150e-07,15.456502,37.358312,1.000000


## Interpretation template

After running the notebook, the decision is:

```text
If rho=1.0 fails:
    Hard projection is unsafe with fixed fractional coordinates.

If rho near the current lambda-equivalent passes:
    Direct soft SG guidance is probably safe at that strength.

If rho near the current lambda-equivalent fails:
    Reduce lambda_sg, use family-specific gating, or prefer an orbit/canonicalization-aware loss.
```

Remember: reducing `proj_pred_k` is not enough. The method is only useful if normal CSP metrics stay healthy.